# 02 · Silver (local + pandas) — limpo, tipado, deduplicado

```
BRONZE (Parquet no bucket warehouse) ──► SILVER (Parquet no bucket warehouse) + QUARENTENA
```

Tudo o que a Silver faz tem uma **regra explícita**. O que viola a regra não some: vai para a **quarentena**.

| Tabela | Regras |
|---|---|
| clientes | `cliente_id` int; `estado` em maiúsculas; e-mail em minúsculas, inválido vira `NULL`; datas em vários formatos; fica a versão mais recente por `cliente_id` |
| produtos | `preco` e `custo` > 0, senão quarentena |
| pedidos | tipos corretos; deduplica por `pedido_id`; **cliente inexistente → quarentena**; `itens` explodem em `itens_pedido` |
| avaliacoes | `nota` entre 1 e 5, senão quarentena; deduplica por `avaliacao_id` |

A Silver é reconstruída a partir de **toda** a Bronze (todos os lotes), então é segura para reexecutar.

**Diferença para a versão Spark:** aqui são arquivos Parquet no bucket, **não tabelas Iceberg**. Não existe `snapshots` nem `VERSION AS OF`. No final usaremos o **versionamento do bucket**, que é o recurso do próprio object storage para guardar o histórico.

**Antes de começar:** rode o `01_bronze_local_pandas.ipynb` (a Bronze precisa existir) com o MinIO no ar.

## Passo 0 · Conexão e funções de apoio

In [1]:
import io
import json
import os
from datetime import datetime, timezone

import boto3
import numpy as np
import pandas as pd

ENDPOINT = os.environ.get("MINIO_ENDPOINT", "http://localhost:9000")
USUARIO = os.environ.get("MINIO_USER", "admin")
SENHA = os.environ.get("MINIO_PASSWORD", "admin12345")

s3 = boto3.client("s3", endpoint_url=ENDPOINT, aws_access_key_id=USUARIO,
                  aws_secret_access_key=SENHA, region_name="us-east-1")


def listar(bucket, prefixo=""):
    achados = []
    for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefixo):
        achados += [(o["Key"], o["Size"]) for o in pagina.get("Contents", [])]
    return achados


def ler_prefixo(bucket, prefixo):
    """Lê todos os Parquet de um prefixo e junta num DataFrame."""
    partes = []
    for chave, _ in listar(bucket, prefixo):
        corpo = s3.get_object(Bucket=bucket, Key=chave)["Body"].read()
        partes.append(pd.read_parquet(io.BytesIO(corpo)))
    return pd.concat(partes, ignore_index=True)


def apagar_prefixo(bucket, prefixo):
    for chave, _ in listar(bucket, prefixo):
        s3.delete_object(Bucket=bucket, Key=chave)


def gravar_silver(df, tabela):
    prefixo = f"silver/{tabela}/"
    apagar_prefixo("warehouse", prefixo)            # reconstruída por inteiro: reexecutar não duplica
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False, engine="pyarrow")
    s3.put_object(Bucket="warehouse", Key=f"{prefixo}part-0000.parquet", Body=buffer.getvalue())
    return len(df)


def para_int(serie):
    return pd.to_numeric(serie, errors="coerce").astype("Int64")


print("conectado em", ENDPOINT)

conectado em http://localhost:9000


## Passo 1 · Ler a Bronze
A Bronze guarda **tudo como veio**: CSVs em texto e pedidos duplicados. A Silver parte daqui.

In [2]:
brz = {t: ler_prefixo("warehouse", f"bronze/{t}/") for t in ("clientes", "produtos", "pedidos", "avaliacoes")}
for t, df in brz.items():
    print(f"{t:<11} {len(df):>7,} linhas | colunas: {list(df.columns)[:8]}")

clientes      5,000 linhas | colunas: ['cliente_id', 'nome', 'email', 'cidade', 'estado', 'data_cadastro', 'canal_aquisicao', '_ingest_ts']
produtos        200 linhas | colunas: ['produto_id', 'nome', 'categoria', 'preco', 'custo', '_ingest_ts', '_source_file', '_batch_id']
pedidos      30,150 linhas | colunas: ['pedido_id', 'cliente_id', 'data_pedido', 'status', 'forma_pagamento', 'valor_frete', 'itens', '_ingest_ts']
avaliacoes    6,489 linhas | colunas: ['avaliacao_id', 'pedido_id', 'cliente_id', 'produto_id', 'nota', 'comentario', 'data_avaliacao', '_ingest_ts']


## Antes da função: faça na mão
Na aula de Spark você explodiu a coluna `itens`. Agora vamos fazer isso **a partir da Bronze**, que tem uma diferença: ela guarda os pedidos duplicados.
Se explodirmos antes de deduplicar, cada item duplicado conta duas vezes e o faturamento fica inflado.

In [3]:
def deduplicar(df, chave):
    """Fica com a versão mais recente (_ingest_ts) de cada chave."""
    return df.sort_values("_ingest_ts").drop_duplicates(chave, keep="last")


def explodir_itens(pedidos):
    """Uma linha por item de pedido. item_seq guarda a posição do item dentro do pedido."""
    base = pedidos[["pedido_id", "cliente_id", "data_pedido", "status", "itens"]].copy()
    base["itens"] = base["itens"].apply(lambda x: list(x) if isinstance(x, (list, np.ndarray)) else [])
    base = base.explode("itens").dropna(subset=["itens"]).reset_index(drop=True)   # explode repete o índice
    base["item_seq"] = base.groupby("pedido_id").cumcount()
    det = pd.DataFrame(base["itens"].tolist()).reindex(columns=["produto_id", "qtd", "preco_unit"])
    itens = base.drop(columns="itens").join(det)
    itens["produto_id"] = para_int(itens["produto_id"])
    itens["qtd"] = pd.to_numeric(itens["qtd"], errors="coerce")
    itens["preco_unit"] = pd.to_numeric(itens["preco_unit"], errors="coerce")
    itens["valor_item"] = (itens["qtd"] * itens["preco_unit"]).round(2)
    return itens


print("Bronze:", len(brz["pedidos"]), "linhas |", brz["pedidos"]["pedido_id"].nunique(), "pedidos distintos")

pedidos_unicos = deduplicar(brz["pedidos"], "pedido_id")
print("Duplicados descartados:", len(brz["pedidos"]) - len(pedidos_unicos))

itens = explodir_itens(pedidos_unicos)
print("pedidos:", len(pedidos_unicos), "| itens:", len(itens))
itens.head(5)

Bronze: 30150 linhas | 30000 pedidos distintos
Duplicados descartados: 150
pedidos: 30000 | itens: 50105


,pedido_id,cliente_id,data_pedido,status,item_seq,produto_id,qtd,preco_unit,valor_item
0,123912,2769,2025-11-28T17:05:06,devolvido,0,6,2,211.62,423.24
1,121529,573,2026-05-22T14:10:13,enviado,0,4,3,805.01,2415.03
2,121529,573,2026-05-22T14:10:13,enviado,1,84,1,228.68,228.68
3,105608,2747,2026-02-14T21:36:17,devolvido,0,60,1,555.50,555.50
4,105608,2747,2026-02-14T21:36:17,devolvido,1,13,1,862.30,862.30


### Checagens de qualidade nos itens
Mesmas três perguntas da aula (valores inválidos, chave duplicada, produto inexistente), mas agora quem falha **não some**: ganha um motivo e vai para a quarentena.

In [4]:
ids_produtos = set(para_int(brz["produtos"]["produto_id"]).dropna())


def separar_itens(itens, ids_produtos):
    valor_ok = (itens["qtd"] > 0) & (itens["preco_unit"] > 0) & itens["produto_id"].notna()
    existe = itens["produto_id"].isin(ids_produtos)
    motivo = np.select([~valor_ok, ~existe], ["valor_invalido", "produto_inexistente"], default="")
    return itens[motivo == ""].copy(), itens[motivo != ""].assign(motivo=motivo[motivo != ""])


itens_ok, itens_rej = separar_itens(itens, ids_produtos)

print("itens válidos:", len(itens_ok), "| rejeitados:", len(itens_rej))
print(itens_rej["motivo"].value_counts().to_string() if len(itens_rej) else "nenhum rejeitado")

# chave composta única (pedido_id, item_seq) nos válidos
dup = int(itens_ok.duplicated(["pedido_id", "item_seq"]).sum())
print("chaves duplicadas:", dup, "->", "OK" if dup == 0 else "ERRO")

itens válidos: 50105 | rejeitados: 0
nenhum rejeitado
chaves duplicadas: 0 -> OK


### Salvar na Silver

In [5]:
n = gravar_silver(itens_ok, "itens_pedido_aula")
print(n, "linhas em s3://warehouse/silver/itens_pedido_aula/")

50105 linhas em s3://warehouse/silver/itens_pedido_aula/


## Passo 2 · O resto da Silver
Agora as outras tabelas, com as mesmas ideias: tipar, padronizar, deduplicar e mandar para a quarentena o que viola a regra.
A função reaproveita `deduplicar`, `explodir_itens` e `separar_itens` que você acabou de escrever.

In [6]:
EMAIL_RE = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"
TS = datetime.now(timezone.utc)


def _json(o):
    return o.tolist() if hasattr(o, "tolist") else str(o)


def para_quarentena(df, origem):
    colunas = ["origem", "registro", "_quarentena_ts"]
    if df.empty:
        return pd.DataFrame(columns=colunas)
    registros = [json.dumps(r, default=_json, ensure_ascii=False) for r in df.to_dict("records")]
    return pd.DataFrame({"origem": origem, "registro": registros, "_quarentena_ts": TS})


def build_silver():
    quarentena = []

    # clientes
    cli = brz["clientes"].copy()
    cli["cliente_id"] = para_int(cli["cliente_id"])
    cli["estado"] = cli["estado"].str.strip().str.upper()
    cli["email"] = cli["email"].str.strip().str.lower()
    cli.loc[~cli["email"].fillna("").str.match(EMAIL_RE), "email"] = None      # inválido vira NULL
    for c in [c for c in cli.columns if c.startswith("data")]:
        cli[c] = pd.to_datetime(cli[c], format="mixed", dayfirst=True, errors="coerce")
    cli = deduplicar(cli, "cliente_id")

    # produtos
    prod = brz["produtos"].copy()
    prod["produto_id"] = para_int(prod["produto_id"])
    for c in ("preco", "custo"):
        prod[c] = pd.to_numeric(prod[c], errors="coerce")
    invalido = ~((prod["preco"] > 0) & (prod["custo"] > 0))
    quarentena.append(para_quarentena(prod[invalido], "produtos"))
    prod = deduplicar(prod[~invalido], "produto_id")

    # pedidos
    ped = deduplicar(brz["pedidos"], "pedido_id").copy()
    ped["cliente_id"] = para_int(ped["cliente_id"])
    ped["data_pedido"] = pd.to_datetime(ped["data_pedido"], format="mixed", errors="coerce")
    orfao = ~ped["cliente_id"].isin(set(cli["cliente_id"].dropna()))
    quarentena.append(para_quarentena(ped[orfao], "pedidos_cliente_orfao"))
    ped = ped[~orfao]

    # itens dos pedidos válidos
    ok, rej = separar_itens(explodir_itens(ped), set(prod["produto_id"].dropna()))
    for motivo, grupo in rej.groupby("motivo"):
        quarentena.append(para_quarentena(grupo.drop(columns="motivo"), f"itens_{motivo}"))
    ped = ped.drop(columns="itens")

    # avaliacoes
    av = brz["avaliacoes"].copy()
    av["nota"] = pd.to_numeric(av["nota"], errors="coerce")
    for c in ("cliente_id", "produto_id"):
        if c in av.columns:
            av[c] = para_int(av[c])
    ruim = ~av["nota"].between(1, 5)
    quarentena.append(para_quarentena(av[ruim], "avaliacoes_nota_invalida"))
    av = deduplicar(av[~ruim], "avaliacao_id")

    tabelas = {"clientes": cli, "produtos": prod, "pedidos": ped, "itens_pedido": ok, "avaliacoes": av,
               "quarentena": pd.concat(quarentena, ignore_index=True)}
    return {nome: gravar_silver(df, nome) for nome, df in tabelas.items()}


resumo = build_silver()
resumo

{'clientes': 5000,
 'produtos': 197,
 'pedidos': 29960,
 'itens_pedido': 49283,
 'avaliacoes': 6464,
 'quarentena': 815}

## Reconciliação: tudo que entrou tem que estar em algum lugar
Regra de ouro: **entrada = saída + quarentena**. Se a conta não fecha, o pipeline está perdendo dado sem avisar.

In [7]:
sil_ped = ler_prefixo("warehouse", "silver/pedidos/")
quar = ler_prefixo("warehouse", "silver/quarentena/")

n_bronze = brz["pedidos"]["pedido_id"].nunique()
n_silver = len(sil_ped)
n_quar = int((quar["origem"] == "pedidos_cliente_orfao").sum())
print(f"pedidos : {n_bronze} = {n_silver} (silver) + {n_quar} (quarentena) -> {n_bronze == n_silver + n_quar}")

n_mao = len(ler_prefixo("warehouse", "silver/itens_pedido_aula/"))
n_funcao = len(ler_prefixo("warehouse", "silver/itens_pedido/"))
print(f"itens   : feito na mão = {n_mao} | feito pela função = {n_funcao}")

pedidos : 30000 = 29960 (silver) + 40 (quarentena) -> True
itens   : feito na mão = 50105 | feito pela função = 49283


**Pergunta:** se os dois números de itens forem diferentes, por quê? Na versão à mão explodimos todos os pedidos; a função só explode os pedidos cujo cliente existe. Os itens dos pedidos órfãos ficaram de fora.

## O que foi para a quarentena?

In [8]:
print(quar["origem"].value_counts().to_string())
quar.head(5)

origem
itens_produto_inexistente    747
pedidos_cliente_orfao         40
avaliacoes_nota_invalida      25
produtos                       3


,origem,registro,_quarentena_ts
0,produtos,"{""produto_id"": 55, ""nome"": ""Jogo de Cama Max"",...",2026-10-26 15:47:41.988631+00:00
1,produtos,"{""produto_id"": 114, ""nome"": ""Sandália Basic"", ...",2026-10-26 15:47:41.988631+00:00
2,produtos,"{""produto_id"": 173, ""nome"": ""Castanha de Caju ...",2026-10-26 15:47:41.988631+00:00
3,pedidos_cliente_orfao,"{""pedido_id"": 122256, ""cliente_id"": 999999, ""d...",2026-10-26 15:47:41.988631+00:00
4,pedidos_cliente_orfao,"{""pedido_id"": 117523, ""cliente_id"": 999999, ""d...",2026-10-26 15:47:41.988631+00:00


## Data Quality: medir, não só limpar
Cada regra devolve uma linha no relatório. Guardar o relatório no storage é o começo da **observabilidade**.

In [9]:
resultados = []
EXEC = datetime.now(timezone.utc)


def regra(df, dataset, nome, coluna, falhou, severidade="erro"):
    n = int(falhou.sum())
    status = "OK" if n == 0 else ("AVISO" if severidade == "aviso" else "ERRO")
    resultados.append({"dataset": dataset, "regra": nome, "coluna": coluna, "total": len(df), "falhas": n,
                       "pct_falha": round(100 * n / max(len(df), 1), 2), "status": status, "executado_em": EXEC})


cli = ler_prefixo("warehouse", "silver/clientes/")
ped = ler_prefixo("warehouse", "silver/pedidos/")
av = ler_prefixo("warehouse", "silver/avaliacoes/")

regra(cli, "silver.clientes", "not_null", "cliente_id", cli["cliente_id"].isna())
regra(cli, "silver.clientes", "unique", "cliente_id", cli["cliente_id"].duplicated())
regra(cli, "silver.clientes", "not_null", "email", cli["email"].isna(), "aviso")
regra(ped, "silver.pedidos", "not_null", "pedido_id", ped["pedido_id"].isna())
regra(ped, "silver.pedidos", "unique", "pedido_id", ped["pedido_id"].duplicated())
regra(ped, "silver.pedidos", "in_set", "status",
      ~ped["status"].isin(["pago", "enviado", "entregue", "cancelado", "devolvido"]))
regra(av, "silver.avaliacoes", "unique", "avaliacao_id", av["avaliacao_id"].duplicated())
regra(av, "silver.avaliacoes", "between", "nota", ~av["nota"].between(1, 5))

relatorio = pd.DataFrame(resultados)

# cada execução grava um objeto NOVO (nunca editamos um objeto existente): o histórico fica no prefixo
chave = f"silver/dq_resultados/{EXEC:%Y%m%dT%H%M%S}.parquet"
buf = io.BytesIO()
relatorio.to_parquet(buf, index=False)
s3.put_object(Bucket="warehouse", Key=chave, Body=buf.getvalue())
print("relatório gravado em", chave, "| execuções guardadas:", len(listar("warehouse", "silver/dq_resultados/")))
relatorio

relatório gravado em silver/dq_resultados/20261026T154742.parquet | execuções guardadas: 1


,dataset,regra,coluna,total,falhas,pct_falha,status,executado_em
0,silver.clientes,not_null,cliente_id,5000,0,0.0,OK,2026-10-26 15:47:42.590635+00:00
1,silver.clientes,unique,cliente_id,5000,0,0.0,OK,2026-10-26 15:47:42.590635+00:00
2,silver.clientes,not_null,email,5000,270,5.4,AVISO,2026-10-26 15:47:42.590635+00:00
3,silver.pedidos,not_null,pedido_id,29960,0,0.0,OK,2026-10-26 15:47:42.590635+00:00
4,silver.pedidos,unique,pedido_id,29960,0,0.0,OK,2026-10-26 15:47:42.590635+00:00
5,silver.pedidos,in_set,status,29960,0,0.0,OK,2026-10-26 15:47:42.590635+00:00
6,silver.avaliacoes,unique,avaliacao_id,6464,0,0.0,OK,2026-10-26 15:47:42.590635+00:00
7,silver.avaliacoes,between,nota,6464,0,0.0,OK,2026-10-26 15:47:42.590635+00:00


Se alguma regra aparece com status `AVISO` ou `ERRO`, ótimo: é o ponto de conversa. (Dica: e-mail inválido vira `NULL` na Silver, e a regra `not_null(email)` aparece como `AVISO`: o cliente continua existindo, só perdeu o contato.)

## Histórico de versões: a Silver tem memória?
Com Iceberg, cada escrita cria um *snapshot*. Aqui não há Iceberg, mas o próprio **object storage** pode guardar versões de cada objeto, quando o **versionamento do bucket** está ligado.
Vamos criar um bucket só para este experimento, ligar o versionamento e repetir o erro humano: alguém "limpa" os pedidos cancelados achando que eram lixo.

In [10]:
BUCKET_H = "historico"
CHAVE_H = "itens_pedido/part-0000.parquet"

if BUCKET_H not in {b["Name"] for b in s3.list_buckets()["Buckets"]}:
    s3.create_bucket(Bucket=BUCKET_H)
s3.put_bucket_versioning(Bucket=BUCKET_H, VersioningConfiguration={"Status": "Enabled"})


def gravar_versao(df):
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    return s3.put_object(Bucket=BUCKET_H, Key=CHAVE_H, Body=buf.getvalue())["VersionId"]


def ler_versao(version_id=None):
    extra = {"VersionId": version_id} if version_id else {}
    corpo = s3.get_object(Bucket=BUCKET_H, Key=CHAVE_H, **extra)["Body"].read()
    return pd.read_parquet(io.BytesIO(corpo))


def historico():
    r = s3.list_object_versions(Bucket=BUCKET_H, Prefix=CHAVE_H)
    return pd.DataFrame([{"version_id": v["VersionId"], "gravado_em": v["LastModified"],
                          "KB": round(v["Size"] / 1024, 1), "atual": v["IsLatest"]}
                         for v in r.get("Versions", [])])      # mais recente primeiro


versao_boa = gravar_versao(itens_ok)
print("linhas na versão boa:", len(itens_ok))
historico()

linhas na versão boa: 50105


,version_id,gravado_em,KB,atual
0,18d646d1-3cf0-43f4-ae23-8fced90e836a,2026-10-26 15:47:42.806000+00:00,898.1,True


Agora o estrago: gravamos de novo o **mesmo objeto**, sem os cancelados.

In [11]:
sem_cancelados = itens_ok[itens_ok["status"] != "cancelado"]
gravar_versao(sem_cancelados)

print(f"linhas: {len(itens_ok)} -> {len(sem_cancelados)}  (sumiram {len(itens_ok) - len(sem_cancelados)})")
print("versão boa:", versao_boa)
historico()

linhas: 50105 -> 43886  (sumiram 6219)
versão boa: 18d646d1-3cf0-43f4-ae23-8fced90e836a


,version_id,gravado_em,KB,atual
0,00f03404-84d6-4070-980c-73d41cdab45b,2026-10-26 15:47:42.861000+00:00,784.0,True
1,18d646d1-3cf0-43f4-ae23-8fced90e836a,2026-10-26 15:47:42.806000+00:00,898.1,False


### Voltar no tempo (só ler)

In [12]:
print("linhas na versão antiga :", len(ler_versao(versao_boa)))
print("linhas na versão atual  :", len(ler_versao()))

linhas na versão antiga : 50105
linhas na versão atual  : 43886


### Desfazer (rollback)

In [13]:
# o rollback é COPIAR a versão boa por cima: isso cria uma versão nova, o histórico só cresce
s3.copy_object(Bucket=BUCKET_H, Key=CHAVE_H,
               CopySource={"Bucket": BUCKET_H, "Key": CHAVE_H, "VersionId": versao_boa})
print("linhas depois do rollback:", len(ler_versao()))
historico()

linhas depois do rollback: 50105


,version_id,gravado_em,KB,atual
0,b0408093-1169-42ec-a1a1-5b7886b55e93,2026-10-26 15:47:42.915000+00:00,898.1,True
1,00f03404-84d6-4070-980c-73d41cdab45b,2026-10-26 15:47:42.861000+00:00,784.0,False
2,18d646d1-3cf0-43f4-ae23-8fced90e836a,2026-10-26 15:47:42.806000+00:00,898.1,False


### O que aprendemos
- A versão errada **não destruiu** a boa: o object storage manteve as duas. Compare com o Passo 5 da Bronze, onde o objeto era sobrescrito e o passado se perdia: a diferença é o versionamento do bucket.
- O rollback aqui não "volta" nada: ele **cria uma nova versão** igual à antiga. Na tabela Iceberg o rollback muda o ponteiro da tabela. Nos dois casos o histórico é preservado.
- Ler uma versão antiga é seguro. Reverter muda o dado para todos, então deve ser feito por quem tem responsabilidade sobre ele.
- **Mas atenção (LGPD):** `delete_object` com versionamento só coloca uma *marca de exclusão*; as versões antigas continuam lá. Para apagar de verdade é preciso apagar cada `VersionId`. **Pergunta:** quanto tempo guardar o histórico? Ele ajuda a auditar, mas atrasa o direito ao esquecimento.
- **Comparação com a versão Spark:** Iceberg versiona a *tabela* (várias linhas e arquivos, de forma consistente). O versionamento do bucket versiona cada *objeto*, um a um, sem saber o que é uma tabela.

## Checkpoint
- [x] Silver com 5 tabelas (`clientes`, `produtos`, `pedidos`, `itens_pedido`, `avaliacoes`) + `quarentena` + `dq_resultados`.
- [x] Você consegue explicar por que um pedido foi para a quarentena.
- [x] Você consegue desfazer uma mudança errada na Silver e explica por que o dado antigo ainda existia.

➡️ Próximo: `03_gold_local_pandas.ipynb`